# CS224N · Lecture 10 — Prompting, Instruction Finetuning, and DPO/RLHF

**Slides:** [cs224n-spr2024-lecture10-prompting-rlhf.pdf](https://web.stanford.edu/class/cs224n/slides/cs224n-spr2024-lecture10-prompting-rlhf.pdf) (Archit Sharma; based on slides from Jesse Mu)  
**Prerequisites:** L9 (pretraining), L8 (Transformers). The NumPy Transformer from L8/L9 is reused.

### From language models to assistants
How do we get from *"Stanford University is located in \_\_\_\_\_"* to an assistant that writes code, explains math, and follows instructions? The lecture's answer has three stages, each fixing a limitation of the previous one:

| Stage | + | − |
|---|---|---|
| **1. Zero/few-shot in-context learning** | no finetuning needed; prompt engineering (e.g., chain of thought) can improve performance | limits to what you can fit in context; complex tasks will probably need gradient steps |
| **2. Instruction finetuning** | simple and straightforward; generalizes to unseen tasks | collecting demonstrations for so many tasks is expensive; mismatch between the LM objective and human preferences |
| **3. Optimizing for human preferences (RLHF / DPO)** | directly models preferences (cf. language modeling); generalizes beyond labeled data | RL is very tricky to get right; human preferences are fallible, models of human preferences even more so |

The slides warn: *"RLHF is still a very underexplored and fast-moving area: by the next lecture these slides may look very different!"* We'll note where practice has moved since (🔎).

| § | Topic | You will implement / measure |
|---|---|---|
| 1 | Emergent zero-shot and few-shot learning (GPT-2, GPT-3) | A tiny Transformer that learns to infer a task from in-context examples, compared with the Bayes-optimal learner |
| 2 | Chain-of-thought prompting, prompt engineering | — |
| 3 | Instruction finetuning | Prompt templates and loss masking |
| 4 | **Reward modeling**: why pairwise comparisons | Ratings vs. comparisons under annotator bias; the Bradley–Terry model |
| 5 | **RLHF**: maximize reward with a KL penalty | Closed-form optimum vs. policy gradient (REINFORCE); **reward over-optimization** |
| 6 | **DPO** | Derivation; DPO recovers the RLHF optimum without a reward model or RL |
| 7 | Limitations and what's next (RLAIF, Constitutional AI, STaR) | — |
| 8 | PyTorch/HF: the DPO loss for a real LM (optional) | — |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random, math, time, collections, itertools

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)
random.seed(0)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

# ---- the NumPy Transformer from L8/L9 ----
def softmax(x, axis=-1):
    z = x - x.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

def ln_forward(x, g, b, eps=1e-5):
    mu = x.mean(-1, keepdims=True); var = x.var(-1, keepdims=True)
    xh = (x - mu) / np.sqrt(var + eps)
    return xh * g + b, (xh, var, eps)

def ln_backward(dy, g, cache):
    xh, var, eps = cache; D = xh.shape[-1]
    dg = (dy * xh).reshape(-1, D).sum(0); db = dy.reshape(-1, D).sum(0)
    dxh = dy * g
    dx = (dxh - dxh.mean(-1, keepdims=True) - xh * (dxh * xh).mean(-1, keepdims=True)) / np.sqrt(var + eps)
    return dx, dg, db

def init_transformer(V, T_max, d=48, n_layers=2, n_heads=4, d_ff=96, seed=0):
    r = np.random.default_rng(seed)
    p = {"tok": r.normal(0, 0.02, (V, d)), "pos": r.normal(0, 0.02, (T_max, d))}
    for l in range(n_layers):
        for k in ["Wq", "Wk", "Wv", "Wo"]:
            p[f"{l}.{k}"] = r.normal(0, 1 / np.sqrt(d), (d, d))
        p[f"{l}.W1"], p[f"{l}.b1"] = r.normal(0, 1 / np.sqrt(d), (d, d_ff)), np.zeros(d_ff)
        p[f"{l}.W2"], p[f"{l}.b2"] = r.normal(0, 1 / np.sqrt(d_ff), (d_ff, d)), np.zeros(d)
        for k in ["ln1", "ln2"]:
            p[f"{l}.{k}g"], p[f"{l}.{k}b"] = np.ones(d), np.zeros(d)
    p["lnfg"], p["lnfb"] = np.ones(d), np.zeros(d)
    return p

def mha_forward(a, Wq, Wk, Wv, Wo, nh, causal, key_mask):
    B, T, d = a.shape; dk = d // nh
    split = lambda z: z.reshape(B, T, nh, dk).transpose(0, 2, 1, 3)
    q, k, v = split(a @ Wq), split(a @ Wk), split(a @ Wv)
    S = q @ k.transpose(0, 1, 3, 2) / np.sqrt(dk)
    if causal:
        S = np.where(np.triu(np.ones((T, T), bool), 1), -1e9, S)
    if key_mask is not None:                                   # (B, T): True = real token
        S = np.where(key_mask[:, None, None, :], S, -1e9)
    P = softmax(S, -1)
    Zc = (P @ v).transpose(0, 2, 1, 3).reshape(B, T, d)
    return Zc @ Wo, (a, q, k, v, P, Zc, dk)

def mha_backward(dout, Wq, Wk, Wv, Wo, nh, cache):
    a, q, k, v, P, Zc, dk = cache; B, T, d = a.shape
    gWo = Zc.reshape(-1, d).T @ dout.reshape(-1, d)
    dZ = (dout @ Wo.T).reshape(B, T, nh, dk).transpose(0, 2, 1, 3)
    dP = dZ @ v.transpose(0, 1, 3, 2); dv = P.transpose(0, 1, 3, 2) @ dZ
    dS = P * (dP - (dP * P).sum(-1, keepdims=True)) / np.sqrt(dk)
    dq, dk_ = dS @ k, dS.transpose(0, 1, 3, 2) @ q
    merge = lambda z: z.transpose(0, 2, 1, 3).reshape(B, T, d)
    dq, dk_, dv = merge(dq), merge(dk_), merge(dv)
    a2 = a.reshape(-1, d)
    return dq @ Wq.T + dk_ @ Wk.T + dv @ Wv.T, (a2.T @ dq.reshape(-1, d), a2.T @ dk_.reshape(-1, d), a2.T @ dv.reshape(-1, d), gWo)

def backbone_forward(p, X, n_layers, n_heads, causal=True, key_mask=None):
    # Pre-LN Transformer. Returns final hidden states (B, T, d) and a cache for the backward pass.
    B, T = X.shape
    x = p["tok"][X] + p["pos"][:T]
    caches, attn = [], []
    for l in range(n_layers):
        a_in, c1 = ln_forward(x, p[f"{l}.ln1g"], p[f"{l}.ln1b"])
        att, ca = mha_forward(a_in, p[f"{l}.Wq"], p[f"{l}.Wk"], p[f"{l}.Wv"], p[f"{l}.Wo"], n_heads, causal, key_mask)
        attn.append(ca[4]); x1 = x + att
        f_in, c2 = ln_forward(x1, p[f"{l}.ln2g"], p[f"{l}.ln2b"])
        hpre = f_in @ p[f"{l}.W1"] + p[f"{l}.b1"]; hid = np.maximum(hpre, 0)
        x = x1 + hid @ p[f"{l}.W2"] + p[f"{l}.b2"]
        caches.append((c1, ca, c2, f_in, hpre, hid))
    xf, cf = ln_forward(x, p["lnfg"], p["lnfb"])
    return xf, (X, caches, cf, attn)

def backbone_backward(p, dxf, cache, n_layers, n_heads):
    X, caches, cf, _ = cache; g = {}; d = dxf.shape[-1]
    dx, g["lnfg"], g["lnfb"] = ln_backward(dxf, p["lnfg"], cf)
    for l in reversed(range(n_layers)):
        c1, ca, c2, f_in, hpre, hid = caches[l]
        g[f"{l}.W2"] = hid.reshape(-1, hid.shape[-1]).T @ dx.reshape(-1, d); g[f"{l}.b2"] = dx.reshape(-1, d).sum(0)
        dhpre = (dx @ p[f"{l}.W2"].T) * (hpre > 0)
        g[f"{l}.W1"] = f_in.reshape(-1, d).T @ dhpre.reshape(-1, dhpre.shape[-1]); g[f"{l}.b1"] = dhpre.reshape(-1, dhpre.shape[-1]).sum(0)
        dx1_ln, g[f"{l}.ln2g"], g[f"{l}.ln2b"] = ln_backward(dhpre @ p[f"{l}.W1"].T, p[f"{l}.ln2g"], c2)
        dx1 = dx + dx1_ln
        da, (g[f"{l}.Wq"], g[f"{l}.Wk"], g[f"{l}.Wv"], g[f"{l}.Wo"]) = mha_backward(
            dx1, p[f"{l}.Wq"], p[f"{l}.Wk"], p[f"{l}.Wv"], p[f"{l}.Wo"], n_heads, ca)
        dxa, g[f"{l}.ln1g"], g[f"{l}.ln1b"] = ln_backward(da, p[f"{l}.ln1g"], c1)
        dx = dx1 + dxa
    g["tok"] = np.zeros_like(p["tok"]); np.add.at(g["tok"], X, dx)
    g["pos"] = np.zeros_like(p["pos"]); g["pos"][:X.shape[1]] = dx.sum(0)
    return g

def token_ce(logits, Y, M):
    # Cross-entropy over positions with mask M. Returns loss, dlogits.
    P = softmax(logits, -1); B, T = Y.shape
    Bi, Ti = np.meshgrid(np.arange(B), np.arange(T), indexing="ij"); n = max(M.sum(), 1)
    loss = -(np.log(P[Bi, Ti, Y] + 1e-12) * M).sum() / n
    G = P.copy(); G[Bi, Ti, Y] -= 1; G *= M[:, :, None] / n
    return loss, G, P

class Adam:
    def __init__(self, params, lr=1e-3, b1=0.9, b2=0.999, eps=1e-8, keys=None):
        self.lr, self.b1, self.b2, self.eps, self.t = lr, b1, b2, eps, 0
        self.keys = list(params) if keys is None else list(keys)          # which parameters to train
        self.m = {k: np.zeros_like(params[k]) for k in self.keys}
        self.v = {k: np.zeros_like(params[k]) for k in self.keys}
    def step(self, params, grads):
        self.t += 1
        for k in self.keys:
            self.m[k] = self.b1 * self.m[k] + (1 - self.b1) * grads[k]
            self.v[k] = self.b2 * self.v[k] + (1 - self.b2) * grads[k] ** 2
            params[k] -= self.lr * (self.m[k] / (1 - self.b1 ** self.t)) / (np.sqrt(self.v[k] / (1 - self.b2 ** self.t)) + self.eps)

def clip_grads(g, max_norm=1.0):
    norm = math.sqrt(sum((v ** 2).sum() for v in g.values()))
    if norm > max_norm:
        for k in g:
            g[k] *= max_norm / norm

---
## 1. Zero-shot and few-shot in-context learning

**Larger and larger models, trained on more and more data.** Language models may even do rudimentary modeling of **agents, beliefs, and actions** (Andreas 2022), and are useful for math, code, and medicine, and as multitask assistants (ChatGPT, Bard/Gemini, Claude).

### 1.1 Emergent abilities: GPT (2018) → GPT-2 (2019) → GPT-3 (2020)
* **GPT** (117M params, 12-layer decoder, BooksCorpus 4.6GB): showed LM pretraining is effective for downstream tasks like NLI, with finetuning (`[START] premise [DELIM] hypothesis [EXTRACT]`).
* **GPT-2** (1.5B params, same architecture, just bigger; 40GB of internet text, **WebText**: links posted on Reddit with at least 3 upvotes, a rough proxy of human quality). Its key emergent ability is **zero-shot learning**: doing many tasks with **no examples and no gradient updates**, by simply
    * **specifying the right sequence prediction problem**, e.g., QA: `Passage: Tom Brady... Q: Where was Tom Brady born? A: ...`
    * **comparing probabilities of sequences**, e.g., Winograd schemas: *The cat couldn't fit into the hat because it was too big.* Is $P(\dots\text{because the cat was too big}) \geq P(\dots\text{because the hat was too big})$? (We did exactly this in L9 §3.2.)
    * GPT-2 beat the state of the art on language-modeling benchmarks like **LAMBADA** (long discourse dependencies) with no task-specific finetuning. You can get interesting zero-shot behavior **if you're creative enough with how you specify the task**: appending **"TL;DR:"** to a news article makes GPT-2 summarize it (not as well as 2018 supervised models trained on 287K examples, but with zero).
* **GPT-3** (175B params; over 600GB of data): **emergent few-shot learning**. Specify a task by simply **prepending examples** of the task before your example. Also called **in-context learning**, stressing that **no gradient updates** are performed.
    * *Zero-shot:* `Translate English to French: cheese =>`
    * *One-shot:* `Translate English to French: sea otter => loutre de mer / cheese =>`
    * *Few-shot:* several such examples, then the query.
* **Few-shot learning is an emergent property of model scale**: on synthetic "word unscrambling" tasks (cycle letters *pleap → apple*, random insertion *a.p!p/l!e → apple*, reversed words *elppa → apple*), accuracy with 100 in-context examples stays near zero for small models and rises sharply for the largest ones.

**Traditional finetuning vs. prompting:** finetuning updates the weights with gradient steps on many examples; zero/few-shot prompting keeps the weights frozen and puts the task description and examples in the input.

### 1.2 🔎 A tiny model that learns to learn in context
How can a model "learn" from examples without gradient steps? One influential view (Xie et al. 2022) is that in-context learning is **implicit Bayesian inference**: pretraining data is a mixture of many "tasks" (documents with latent topics/styles), so predicting the next token *requires* inferring which task you're in from the context so far. The examples in a prompt are evidence about that latent task.

We can test this at toy scale. Each training sequence uses a **hidden function** $f$, drawn from four:
$$f(x) = x,\qquad f(x) = x+1 \bmod 10,\qquad f(x) = 9-x,\qquad f(x) = 2x \bmod 10$$
and shows examples $x_1\ f(x_1)\ x_2\ f(x_2)\ \dots$ The model is trained as a plain language model: predict each $f(x_i)$ from everything before it. It's never told which function is in use. To predict well, it must **infer $f$ from the earlier examples in its context**.

Because we know the generating process, we can compute the **Bayes-optimal** prediction exactly: the posterior over the four functions given the examples so far, and the most probable answer under it.

In [ ]:
TASKS = {"identity": lambda x: x, "plus one": lambda x: (x + 1) % 10,
         "nine minus": lambda x: 9 - x, "times two": lambda x: (2 * x) % 10}
TASK_FNS = list(TASKS.values())
BOS, N_EX = 10, 8                       # tokens 0-9 are digits, 10 = BOS; 8 examples per sequence
V_ICL, T_ICL = 11, 2 * N_EX              # input = BOS x1 f(x1) ... x8  (16 tokens); targets shifted by one

def icl_batch(n, r, fns=TASK_FNS):
    X = np.zeros((n, T_ICL), int); Y = np.zeros((n, T_ICL), int); M = np.zeros((n, T_ICL))
    for b in range(n):
        f = fns[r.integers(len(fns))]                                 # the hidden task for this sequence
        xs = r.integers(0, 10, N_EX)
        seq = [BOS] + [t for x in xs for t in (int(x), int(f(x)))]    # BOS x1 f(x1) x2 f(x2) ...
        X[b], Y[b] = seq[:-1], seq[1:]                                # next-token prediction
        M[b, 1::2] = 1                                                # loss only where the target is an answer f(x_i)
    return X, Y, M

r = np.random.default_rng(0)
X_demo, Y_demo, M_demo = icl_batch(2, r)
print("input  :", X_demo[0]); print("targets:", np.where(M_demo[0] > 0, Y_demo[0], -1), "(-1 = no loss)")

In the input, position $2i-1$ holds $x_i$ and the model must predict $f(x_i)$ there. So at the $i$-th answer, the context contains $i-1$ complete examples: **answer 1 is "zero-shot"** (no examples yet), answer 2 is "one-shot," and so on.

In [ ]:
N_LAYERS, N_HEADS = 2, 2
p_icl = init_transformer(V_ICL, T_ICL, d=32, n_layers=N_LAYERS, n_heads=N_HEADS, d_ff=64, seed=0)
p_icl["Wout"] = np.random.default_rng(0).normal(0, 1 / np.sqrt(32), (32, V_ICL))

def icl_loss_and_grads(p, X, Y, M):
    xf, cache = backbone_forward(p, X, N_LAYERS, N_HEADS, causal=True)
    loss, G, P = token_ce(xf @ p["Wout"], Y, M)
    g = backbone_backward(p, G @ p["Wout"].T, cache, N_LAYERS, N_HEADS)
    g["Wout"] = xf.reshape(-1, xf.shape[-1]).T @ G.reshape(-1, G.shape[-1])
    return loss, g, P

opt = Adam(p_icl, lr=2e-3); r = np.random.default_rng(0); t0 = time.time()
for step in range(1, 2001):
    X, Y, M = icl_batch(64, r)
    loss, g, _ = icl_loss_and_grads(p_icl, X, Y, M); clip_grads(g); opt.step(p_icl, g)
    if step % 500 == 0:
        print(f"step {step}: loss on answers {loss:.3f}  ({time.time() - t0:.0f}s)")

In [ ]:
def bayes_optimal_accuracy(X, Y, fns=TASK_FNS):
    # For each answer position, predict argmax_y sum_f P(f | previous examples) [f(x) = y]; uniform prior over fns.
    n = len(X); acc = np.zeros(N_EX)
    for b in range(n):
        alive = np.ones(len(fns), bool)
        for i in range(N_EX):
            x, y = X[b, 2 * i + 1], Y[b, 2 * i + 1]
            votes = collections.Counter(fns[k](x) for k in range(len(fns)) if alive[k])
            acc[i] += votes.most_common(1)[0][0] == y if votes else 0
            alive &= np.array([fns[k](x) == y for k in range(len(fns))])     # observe the true answer
    return acc / n

def model_accuracy(p, X, Y, M):
    _, _, P = icl_loss_and_grads(p, X, Y, M)
    pred = P.argmax(-1)
    return np.array([(pred[:, 2 * i + 1] == Y[:, 2 * i + 1]).mean() for i in range(N_EX)])

r = np.random.default_rng(123)
X, Y, M = icl_batch(2000, r)
acc_model, acc_bayes = model_accuracy(p_icl, X, Y, M), bayes_optimal_accuracy(X, Y)
X5, Y5, M5 = icl_batch(2000, r, fns=[lambda x: (x + 3) % 10])            # a NEW function, never seen in training
acc_new = model_accuracy(p_icl, X5, Y5, M5)
print("examples already in context:", list(range(N_EX)))
print("transformer, training tasks:", acc_model.round(3))
print("Bayes-optimal learner       :", acc_bayes.round(3))
print("transformer, unseen x+3 task:", acc_new.round(3))

plt.figure(figsize=(7, 4))
plt.plot(range(N_EX), acc_model, "o-", label="Transformer (training tasks)")
plt.plot(range(N_EX), acc_bayes, "k--", label="Bayes-optimal learner")
plt.plot(range(N_EX), acc_new, "s-", label="Transformer, unseen task f(x)=x+3")
plt.xlabel("number of examples in the context ('k-shot')"); plt.ylabel("accuracy on the next answer")
plt.ylim(0, 1.05); plt.legend(); plt.title("In-context learning without gradient steps"); plt.show()

**What this shows.** The model was trained only to predict the next token, yet at test time it **gets better with every example in its context, with no gradient steps**. That's in-context learning, with a zero-shot → one-shot → few-shot curve like GPT-3's. And it tracks the **Bayes-optimal** learner closely: it has learned to infer which latent task generated the context.

The unseen task $f(x) = x+3$ exposes the limit at this scale. The model only learned to **pick among the tasks it saw during training**; it can't induce a genuinely new rule from examples. Whether (and how far) large LMs go beyond "task location" to real in-context generalization is an active research question. Larger models trained on more diverse tasks do much better (e.g., Garg et al. 2022 on in-context linear regression; Raventós et al. 2023 show a transition from task retrieval to genuine in-context learning as task diversity grows).

---
## 2. Limits of prompting for harder tasks? Chain-of-thought

Some tasks seem too hard for even large LMs to learn through prompting alone, especially tasks involving **richer, multi-step reasoning** (humans struggle at these too!):

```
19583 + 29534 = 49117
98394 + 49384 = 147778
29382 + 12347 = 41729
93847 + 39299 = ?
```

**Solution: change the prompt!**

### 2.1 Chain-of-thought (CoT) prompting (Wei et al. 2022; also Nye et al. 2021)
Give few-shot examples whose answers include the **reasoning steps**, not just the final answer. The model then generates its own reasoning before answering. *Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now? A: Roger started with 5 balls. 2 cans of 3 tennis balls each is 6 tennis balls. 5 + 6 = 11. The answer is 11.*

**Chain-of-thought prompting is an emergent property of model scale:** on middle-school math word problems (GSM8K), CoT helps little or even hurts for small models, and helps dramatically for the largest ones.

### 2.2 Zero-shot chain-of-thought (Kojima et al. 2022)
*Do we even need examples of reasoning? Can we just ask the model to reason through things?*

`Q: A juggler can juggle 16 balls. Half of the balls are golf balls, and half of the golf balls are blue. How many blue golf balls are there? A: Let's think step by step.` → *There are 16 balls in total. Half of the balls are golf balls. That means there are 8 golf balls. Half of the golf balls are blue. That means there are 4 blue golf balls.*

Zero-shot CoT **greatly outperforms zero-shot**; manual (few-shot) CoT is still better. An **LM-designed** prompt, *"Let's work this out in a step by step way to be sure we have the right answer"* (Zhou et al. 2022), did even better.

🔎 **Why might CoT help?** A Transformer does a fixed amount of computation per generated token. Writing out intermediate steps lets the model spend more computation on a problem, and store intermediate results in its own context (a "scratchpad"). Today's reasoning models are trained with RL to produce long chains of thought, rather than relying on a prompt to elicit them (🔎 §7).

### 2.3 The new dark art of "prompt engineering"?
* Asking a model for reasoning ("Let's think step by step")
* "Jailbreaking" LMs (e.g., instructing the model to ignore its previous instructions)
* Using a Google code header to generate more "professional" code
* Text-to-image prompts like *"fantasy concept art, glowing blue dodecahedron die on a wooden table, in a cozy fantasy (workshop), tools on the table, artstation, depth of field, 4k, masterpiece"*

---
## 3. Instruction finetuning

### 3.1 Language modeling ≠ assisting users
Language models are **not aligned with user intent** (Ouyang et al. 2022). Asked to *"Explain the moon landing to a 6 year old in a few sentences,"* a base GPT-3 might continue with more similar instructions (*"Explain the theory of gravity to a 6 year old. Explain the theory of relativity..."*), because that's a plausible *continuation* of a list of prompts on the internet. **Finetuning to the rescue!** A human-written demonstration answers instead: *"A giant rocket ship blasted off from Earth carrying astronauts to the moon..."*

### 3.2 Scaling up finetuning
Recall the pretraining/finetuning paradigm (pretrain on LM → finetune on your task). **Instruction finetuning** finetunes on **many tasks** at once: collect examples of **(instruction, output)** pairs across many tasks and finetune an LM, then **evaluate on unseen tasks** (FLAN-T5; Chung et al. 2022).
* As is usually the case, **data + model scale is key**. E.g., the **Super-NaturalInstructions** dataset contains over **1.6K tasks, 3M+ examples**: classification, sequence tagging, rewriting, translation, QA... (Wang et al. 2022).
* **Flan-T5:** T5 models finetuned on 1.8K additional tasks. **Bigger model = bigger gain** (on BIG-bench + MMLU average). Before instruction finetuning, T5 answers a reasoning question with a non-answer; after, it reasons step by step and answers.

**Aside: benchmarks for multitask LMs.**
* **MMLU** (Massive Multitask Language Understanding; Hendrycks et al. 2021): 57 diverse knowledge-intensive tasks (from elementary math to law and medicine). Progress on it has been rapid and impressive.
* **BIG-Bench** (Srivastava et al. 2022): 200+ tasks.
* L11 covers evaluation in depth.

**A huge diversity of instruction-tuning datasets** appeared after the release of LLaMA (Alpaca, Dolly, OpenAssistant, ...). **Lessons:** you can **generate data synthetically** (from bigger LMs); you **don't need many samples** to instruction-tune (e.g., LIMA: 1,000 carefully curated examples); and **crowdsourcing** can be pretty effective.

### 3.3 What instruction-tuning data looks like to the model
Each example is rendered into a **template**, and the loss is computed only on the **response** tokens: we want the model to learn to *answer*, not to generate instructions. (Chat models use the same idea with role markers like `<|user|>` / `<|assistant|>`, and multi-turn conversations.)

In [ ]:
TEMPLATE = "### Instruction:\n{instruction}\n\n### Response:\n"
examples = [("Translate to French: cheese", "fromage"),
            ("Is this review positive or negative? 'The plot was dull.'", "negative")]

for instruction, response in examples:
    prompt = TEMPLATE.format(instruction=instruction)
    tokens = prompt.split() + response.split() + ["<eos>"]
    n_prompt = len(prompt.split())
    loss_mask = [0] * n_prompt + [1] * (len(tokens) - n_prompt)
    print(" ".join(f"{t}" + ("*" if m else "") for t, m in zip(tokens, loss_mask)))
print("\n(* = token contributes to the loss; the instruction tokens are context only)")

### 3.4 Limitations of instruction finetuning
One limitation is obvious: it's **expensive to collect ground-truth data** for tasks. Subtler ones:
* **Problem 1:** tasks like open-ended creative generation have **no right answer**. (*Write me a story about a dog and her pet grasshopper.*)
* **Problem 2:** language modeling **penalizes all token-level mistakes equally**, but some errors are worse than others. If the reference is *"Avatar is a fantasy TV show"*, predicting *"adventure"* instead of *"fantasy"* is a small mistake, and *"musical"* is a big one. Cross-entropy only knows "not the reference token."
* **Problem 3:** **humans generate suboptimal answers**.

Even with instruction finetuning, there's a **mismatch between the LM objective and the objective of "satisfy human preferences"!** Can we explicitly attempt to satisfy human preferences?

---
## 4. Optimizing for human preferences: reward models

Say we're training an LM on some task (e.g., summarization). For an instruction $x$ and an LM sample $y$, imagine we had a way to obtain a **human reward** for that summary: $R(x, y)\in\mathbb{R}$, higher is better. E.g., for a news article about an earthquake: *"An earthquake hit San Francisco. There was minor property damage, but no injuries."* gets $R = 8.0$; *"The Bay Area has good weather but is prone to earthquakes and wildfires."* gets $R = 1.2$. Now we want to **maximize the expected reward of samples from our LM**:
$$\mathbb{E}_{\hat y\sim p_\theta(y\mid x)}\big[R(x, \hat y)\big]$$

**The high-level RLHF pipeline** (Ouyang et al. 2022, InstructGPT): **first step: instruction tuning!** Second and third steps: **maximize reward** (but how??).

### 4.1 How do we get the rewards?
**Problem 1: a human in the loop is expensive!** **Solution:** instead of directly asking humans for preferences, **model their preferences** as a separate (NLP) problem (Knox & Stone 2009): train a **reward model** $RM_\phi(x, y)$ to predict human reward from an annotated dataset, then optimize for $RM_\phi$ instead.

**Problem 2: human judgments are noisy and miscalibrated!** Is a summary *"A 4.2 magnitude earthquake hit San Francisco, resulting in massive damage"* worth 4.1? 6.6? 3.2? **Solution:** instead of asking for direct ratings, ask for **pairwise comparisons**, which can be more reliable (Phelps et al. 2015; Clark et al. 2018): $y_1 > y_3 > y_2$.

### 4.2 Why comparisons? A simulation
Each annotator has their own **offset** (some are generous, some harsh) and **scale** (some use the whole 1–10 range, some don't), plus noise. With direct ratings, those personal quirks pollute the data. With comparisons, an offset cancels out entirely (it's added to both items), and a scale only changes how confident the judgment is. Below, 40 annotators judge 200 responses with known true quality, with the **same annotation budget** for both protocols. We recover a score per response either by averaging ratings, or by fitting the **Bradley–Terry model** (next section) to comparisons.

In [ ]:
r = np.random.default_rng(0)
n_items, n_annot, budget = 200, 40, 2000
q_true = r.normal(size=n_items)
offset, scale = r.normal(0, 1.5, n_annot), r.uniform(0.3, 2.0, n_annot)

# Protocol A: direct ratings. Each judgment = one annotator rates one item.
ann = r.integers(0, n_annot, budget); item = r.integers(0, n_items, budget)
ratings = offset[ann] + scale[ann] * q_true[item] + r.normal(0, 0.5, budget)
sums, cnt = np.zeros(n_items), np.zeros(n_items)
np.add.at(sums, item, ratings); np.add.at(cnt, item, 1)
score_ratings = sums / np.maximum(cnt, 1)

# Protocol B: pairwise comparisons. Each judgment = one annotator compares two items.
ann = r.integers(0, n_annot, budget); i_, j_ = r.integers(0, n_items, budget), r.integers(0, n_items, budget)
util_i = offset[ann] + scale[ann] * q_true[i_] + r.normal(0, 0.5, budget)
util_j = offset[ann] + scale[ann] * q_true[j_] + r.normal(0, 0.5, budget)
win = util_i > util_j
winners, losers = np.where(win, i_, j_), np.where(win, j_, i_)

def fit_bradley_terry(winners, losers, n, steps=500, lr=0.5, l2=1e-3):
    s = np.zeros(n)
    for _ in range(steps):           # maximize sum log sigmoid(s_w - s_l)  (gradient ascent)
        g_ = 1 - sigmoid(s[winners] - s[losers])
        grad = np.zeros(n); np.add.at(grad, winners, g_); np.add.at(grad, losers, -g_)
        s += lr * (grad / len(winners) * n / 10 - l2 * s)
    return s - s.mean()
score_bt = fit_bradley_terry(winners, losers, n_items)

from scipy.stats import spearmanr
print(f"Spearman correlation with true quality, same budget of {budget} judgments:")
print(f"  averaged direct ratings      : {spearmanr(score_ratings, q_true).correlation:.3f}")
print(f"  Bradley-Terry on comparisons : {spearmanr(score_bt, q_true).correlation:.3f}")

### 4.3 The reward model: Bradley–Terry (1952) paired comparisons
Train $RM_\phi$ so that the "winning" sample $y^w$ scores higher than the "losing" sample $y^l$:
$$J_{RM}(\phi) = -\mathbb{E}_{(x, y^w, y^l)\sim D}\Big[\log\sigma\big(RM_\phi(x, y^w) - RM_\phi(x, y^l)\big)\Big]$$
This is the Bradley–Terry model: $P(y^w\succ y^l) = \sigma(r_w - r_l)$. It's just **logistic regression on reward differences**, and only differences matter (adding a constant to all rewards changes nothing).

In practice, $RM_\phi$ is a pretrained LM with a scalar head. Here we use a simple **linear reward model** on two observable features of each response, which lets us see exactly what it learns. Our setup for the rest of the lecture:
* 30 **prompts**, each with 10 candidate **responses** (think: samples from the instruction-tuned model). Each response has a **true quality** $q$ (what humans actually care about) and a **length** $\ell$.
* In typical responses, longer ones tend to be better (they include more detail), so length correlates with quality.
* But each prompt also has one **"verbose junk"** response: very long, low quality, and rarely produced by the reference model.
* The reward model sees two features: a **noisy quality signal** $s = q + \text{noise}$ and the **length** $\ell$. The human preference labels come from the true quality, via Bradley–Terry.

In [ ]:
r = np.random.default_rng(1)
N_PROMPTS, K = 30, 10
q = r.normal(size=(N_PROMPTS, K))                                     # true quality (what humans care about)
length = 0.8 * q + r.normal(0, 0.6, (N_PROMPTS, K))                   # longer responses tend to be better...
q[:, -1] = -1.5; length[:, -1] = 4.0                                  # ...except one 'verbose junk' response per prompt
signal = q + r.normal(0, 1.0, (N_PROMPTS, K))                         # the reward model's noisy view of quality
features = np.stack([signal, length], -1)                            # (prompts, responses, 2)

ref_logits = r.normal(0, 0.5, (N_PROMPTS, K)); ref_logits[:, -1] = -3.0   # reference model rarely produces junk
p_ref = np.exp(ref_logits) / np.exp(ref_logits).sum(1, keepdims=True)

# Preference dataset: pairs sampled from the reference model; labels from humans (Bradley-Terry on true quality)
def sample_preferences(n_pairs, r):
    data = []
    for _ in range(n_pairs):
        x = r.integers(N_PROMPTS)
        a, b = r.choice(K, 2, replace=False, p=p_ref[x])
        w, l = (a, b) if r.random() < sigmoid(q[x, a] - q[x, b]) else (b, a)
        data.append((x, w, l))
    return np.array(data)
prefs = sample_preferences(3000, r)
print(f"{len(prefs)} comparisons; fraction involving the junk response: {np.mean((prefs[:, 1] == K - 1) | (prefs[:, 2] == K - 1)):.2%}")

# Train the linear reward model RM(x, y) = w . features[x, y]  with the Bradley-Terry loss
w_rm = np.zeros(2)
for step in range(2000):
    d = features[prefs[:, 0], prefs[:, 1]] - features[prefs[:, 0], prefs[:, 2]]     # feature differences
    pr = sigmoid(d @ w_rm)
    w_rm += 0.5 * ((1 - pr)[:, None] * d).mean(0)                                    # gradient ascent on log sigma
RM = features @ w_rm
print(f"learned reward model weights: signal {w_rm[0]:.3f}, length {w_rm[1]:.3f}")
print(f"RM score of the junk response vs. the best real response (avg over prompts): "
      f"{RM[:, -1].mean():.2f} vs {RM[:, :-1].max(1).mean():.2f}")

The reward model gives **positive weight to length**, which is a reasonable inference from the data it saw: among normal responses, longer *is* better. But it rarely saw the junk response, so it can't know that its extreme length is a bad sign. That's a **misspecified reward**: correct on-distribution, wrong off-distribution.

---
## 5. RLHF: optimizing the learned reward model

We have a pretrained (possibly instruction-finetuned) LM $p^{PT}(y\mid x)$ and a reward model $RM_\phi(x, y)$ trained on human comparisons. Now to do RLHF: copy the model to $p^{RL}_\theta(y\mid x)$, with parameters $\theta$ to optimize, and maximize
$$\mathbb{E}_{\hat y\sim p^{RL}_\theta(\hat y\mid x)}\big[RM_\phi(x, \hat y)\big]$$
**Do you see any problems?** Learned rewards are imperfect, so this quantity can be imperfectly optimized. **Add a penalty for drifting too far from the initialization:**
$$\mathbb{E}_{\hat y\sim p^{RL}_\theta(\hat y\mid x)}\Big[RM_\phi(x, \hat y) - \beta\log\frac{p^{RL}_\theta(\hat y\mid x)}{p^{PT}(\hat y\mid x)}\Big]$$
We pay a price when $p^{RL}_\theta(\hat y\mid x) > p^{PT}(\hat y\mid x)$. In expectation, this penalty is the **Kullback–Leibler (KL) divergence** between $p^{RL}_\theta(\hat y\mid x)$ and $p^{PT}(\hat y\mid x)$.

### 5.1 How to optimize? Reinforcement learning!
The field of RL has studied these problems for many years (Williams 1992; Sutton & Barto 1998); deep RL resurged circa 2013 (Mnih et al., game playing); applying RL to modern LMs is newer (Ziegler et al. 2019; Stiennon et al. 2020; Ouyang et al. 2022). The general idea:
* **Generate completions** from $p^{RL}_\theta$ for several tasks
* **Compute the reward** using $RM_\phi(x, y)$
* **Update** $p^{RL}_\theta(y\mid x)$ to increase the probability of high-reward completions

The simplest such algorithm is **REINFORCE** (the policy gradient): $\nabla_\theta\,\mathbb{E}_{y\sim p_\theta}[R(y)] = \mathbb{E}_{y\sim p_\theta}\big[R(y)\,\nabla_\theta\log p_\theta(y)\big]$, estimated from samples, usually with a baseline subtracted from $R$ to reduce variance. Production RLHF used **PPO** (Schulman et al. 2017), which adds a learned value function and clipped updates; it's computationally expensive and tricky ("Secrets of RLHF", Zheng et al. 2023): fitting a value function, slow online sampling, and sensitivity to hyperparameters.

### 5.2 The closed-form optimum
For a fixed prompt, the KL-regularized objective has an exact solution (we'll derive it in §6):
$$p^*(y\mid x) = \frac{1}{Z(x)}\,p^{PT}(y\mid x)\,\exp\!\Big(\frac{1}{\beta}RM(x, y)\Big)$$
Our "LM" here is a policy over 10 responses per prompt, so we can compute $p^*$ exactly and check that REINFORCE, which only sees *samples*, converges to it:

In [ ]:
def rlhf_optimum(reward, beta):
    logits = np.log(p_ref) + reward / beta
    return np.exp(logits - logits.max(1, keepdims=True)) / np.exp(logits - logits.max(1, keepdims=True)).sum(1, keepdims=True)

def reinforce(reward, beta, steps=3000, n_samples=32, lr=0.5, seed=0):
    r = np.random.default_rng(seed)
    theta = np.log(p_ref).copy()                                    # initialize the policy at the reference model
    for _ in range(steps):
        pol = np.exp(theta - theta.max(1, keepdims=True)); pol /= pol.sum(1, keepdims=True)
        grad = np.zeros_like(theta)
        for x in range(N_PROMPTS):
            ys = r.choice(K, n_samples, p=pol[x])
            R = reward[x, ys] - beta * (np.log(pol[x, ys]) - np.log(p_ref[x, ys]))      # KL-penalized reward
            A_ = R - R.mean()                                                           # baseline
            for y_, a_ in zip(ys, A_):
                grad[x] += a_ * ((np.arange(K) == y_) - pol[x])                         # a * grad log pi(y)
        theta += lr * grad / n_samples
    pol = np.exp(theta - theta.max(1, keepdims=True)); return pol / pol.sum(1, keepdims=True)

beta = 0.5
p_star = rlhf_optimum(RM, beta)
p_pg = reinforce(RM, beta, steps=400)
kl = lambda p, q_: (p * (np.log(p + 1e-12) - np.log(q_ + 1e-12))).sum(1).mean()
print(f"KL(REINFORCE policy || closed-form optimum) = {kl(p_pg, p_star):.4f}   (vs. KL(reference || optimum) = {kl(p_ref, p_star):.3f})")

### 5.3 Reward over-optimization ("reward hacking")
How strongly should we optimize? The KL weight $\beta$ controls it: small $\beta$ means "maximize the reward model at almost any cost." We sweep $\beta$ and track, for each resulting policy, its KL from the reference model, the **reward model's** score (what we optimize), and the **true quality** (what humans want). This reproduces the shape of Stiennon et al. (2020) and Gao et al. (2022).

In [ ]:
betas = np.logspace(1.5, -2, 40)
rows = []
for b_ in betas:
    pol = rlhf_optimum(RM, b_)
    rows.append((kl(pol, p_ref), (pol * RM).sum(1).mean(), (pol * q).sum(1).mean(), pol[:, -1].mean()))
rows = np.array(rows)
gold = np.array([(kl(rlhf_optimum(q, b_), p_ref), (rlhf_optimum(q, b_) * q).sum(1).mean()) for b_ in betas])

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(rows[:, 0], rows[:, 1], label="reward model score (proxy, optimized)")
ax[0].plot(rows[:, 0], rows[:, 2], label="true quality of the RLHF policy")
ax[0].plot(gold[:, 0], gold[:, 1], "k:", label="true quality, if optimizing the TRUE reward")
ax[0].set_xlabel("KL(policy || reference)"); ax[0].legend(fontsize=8); ax[0].set_title("Reward over-optimization")
ax[1].plot(rows[:, 0], rows[:, 3]); ax[1].set_xlabel("KL(policy || reference)"); ax[1].set_ylabel("P(verbose junk)")
ax[1].set_title("What the policy learns to exploit"); plt.tight_layout(); plt.show()
best = rows[:, 2].argmax()
print(f"true quality peaks at KL = {rows[best, 0]:.2f} (beta = {betas[best]:.3f}): quality {rows[best, 2]:.3f}")
print(f"at the smallest beta (KL = {rows[-1, 0]:.2f}): proxy reward {rows[-1, 1]:.2f}, true quality {rows[-1, 2]:.3f}, P(junk) = {rows[-1, 3]:.2f}")
print(f"reference model: true quality {(p_ref * q).sum(1).mean():.3f}")

At first, optimizing the reward model improves true quality: it shifts probability toward genuinely better responses. Push harder (smaller $\beta$, larger KL) and the **proxy reward keeps climbing while true quality falls**, as the policy piles probability onto the verbose junk the reward model mistakenly loves. This is **Goodhart's law**: *when a measure becomes a target, it ceases to be a good measure.* The KL penalty is what keeps RLHF in the region where the reward model is trustworthy, and **early stopping / choosing $\beta$** is a crucial (and empirical) part of RLHF.

The slides' examples of this in the wild:
* **"Reward hacking" is a common problem in RL** (OpenAI's boat-racing agent that loops forever collecting points instead of finishing the race).
* **Chatbots are rewarded to produce responses that *seem* authoritative and helpful, regardless of truth.** This can result in making up facts and hallucinations (e.g., Bard's factual error in its launch demo).
* **Models of human preferences are even more unreliable** than the preferences themselves.
* There is a real concern of **AI (mis)alignment**.
* 🔎 Our junk response is a cartoon of a real, measured bias: **RLHF'd models are longer** (and reward models and LLM judges prefer longer answers, L11). The slides show the "clear stylistic changes" from RLHF/DPO: significantly more detailed, list-like formatting (Dubois et al. 2023).

**RLHF provides gains over pretraining + finetuning** when done well: in Stiennon et al. (2020), RLHF summaries were preferred by humans over supervised-finetuned ones, and even over the human-written reference summaries.

---
## 6. Direct Preference Optimization (DPO)

The current pipeline: (1) train a reward model on human comparisons; (2) optimize the LM against it with RL. **What if there was a way to write $RM_\phi(x, y)$ in terms of $p^{RL}_\theta(\hat y\mid x)$?** Then we could optimize $\theta$ by fitting the reward model directly to the preference data. How is this possible? **The only external information to the optimization comes from the preference labels.**

### 6.1 Derivation
**Step 1: the closed form.** We want to maximize $\mathbb{E}_{\hat y\sim p^{RL}_\theta}\big[RM(x,\hat y)\big] - \beta\,\text{KL}\big(p^{RL}_\theta\,\|\,p^{PT}\big)$. Write it as
$$\sum_y p(y)\Big[RM(y) - \beta\log\frac{p(y)}{p^{PT}(y)}\Big] = -\beta\sum_y p(y)\log\frac{p(y)}{p^{PT}(y)\,e^{RM(y)/\beta}} = -\beta\,\text{KL}\Big(p\,\Big\|\,\tfrac{1}{Z}p^{PT}e^{RM/\beta}\Big) + \beta\log Z$$
The KL is minimized (= 0) exactly when $p$ equals the normalized distribution:
$$p^*(\hat y\mid x) = \frac{1}{Z(x)}\,p^{PT}(\hat y\mid x)\,\exp\Big(\frac1\beta RM(x,\hat y)\Big)$$

**Step 2: rearrange the terms** to express the reward through the optimal policy:
$$RM(x,\hat y) = \beta\log\frac{p^*(\hat y\mid x)}{p^{PT}(\hat y\mid x)} + \beta\log Z(x)$$
This holds for **arbitrary** LMs, so we can **define** a reward model by a policy: $RM_\theta(x,\hat y) = \beta\log\frac{p^{RL}_\theta(\hat y\mid x)}{p^{PT}(\hat y\mid x)} + \beta\log Z(x)$.

**Step 3: plug into the reward-model loss.** Recall how we fit the reward model: $J_{RM}(\phi) = -\mathbb{E}\big[\log\sigma(RM_\phi(x,y^w) - RM_\phi(x,y^l))\big]$. Notice that we **only need the difference** between the rewards for $y^w$ and $y^l$, so the intractable $\beta\log Z(x)$ **cancels**:
$$RM_\theta(x,y^w) - RM_\theta(x,y^l) = \beta\log\frac{p^{RL}_\theta(y^w\mid x)}{p^{PT}(y^w\mid x)} - \beta\log\frac{p^{RL}_\theta(y^l\mid x)}{p^{PT}(y^l\mid x)}$$
**The final DPO loss:**
$$\boxed{J_{\text{DPO}}(\theta) = -\mathbb{E}_{(x,y^w,y^l)\sim D}\Big[\log\sigma\Big(\beta\log\frac{p^{RL}_\theta(y^w\mid x)}{p^{PT}(y^w\mid x)} - \beta\log\frac{p^{RL}_\theta(y^l\mid x)}{p^{PT}(y^l\mid x)}\Big)\Big]}$$
**A simple classification loss function that connects preference data to language model parameters directly!** For a real LM, $\log p_\theta(y\mid x) = \sum_t\log p_\theta(y_t\mid x, y_{<t})$, computed with one forward pass, like supervised finetuning.

### 6.2 DPO recovers the RLHF optimum
Theory says: if preferences come from Bradley–Terry with reward $r$, then with enough data, DPO's minimizer is $p^{PT}e^{r/\beta}/Z$, the **same** policy as RLHF with the *true* reward. No reward model, no sampling, no RL. Let's check, with a tabular policy (one logit per response):

In [ ]:
def train_dpo(pref_data, beta, steps=1500, lr=1.0):
    theta = np.log(p_ref).copy()
    x, w, l = pref_data[:, 0], pref_data[:, 1], pref_data[:, 2]
    for _ in range(steps):
        logp = theta - np.log(np.exp(theta).sum(1, keepdims=True))
        u = beta * ((logp[x, w] - np.log(p_ref[x, w])) - (logp[x, l] - np.log(p_ref[x, l])))
        coef = (1 - sigmoid(u)) * beta                                  # d(-log sigma(u))/du = -(1 - sigma(u))
        grad = np.zeros_like(theta)                                     # d u / d theta = beta (e_w - e_l): the softmax terms cancel
        np.add.at(grad, (x, w), coef); np.add.at(grad, (x, l), -coef)
        theta += lr * grad / len(pref_data) * N_PROMPTS
    return np.exp(theta) / np.exp(theta).sum(1, keepdims=True)

# Preference data covering all response pairs (DPO's guarantee needs coverage), labels from Bradley-Terry on true q
r = np.random.default_rng(5); data = []
for x in range(N_PROMPTS):
    for a, b in itertools.combinations(range(K), 2):
        for _ in range(30):
            data.append((x, a, b) if r.random() < sigmoid(q[x, a] - q[x, b]) else (x, b, a))
data = np.array(data)

beta = 1.0
p_dpo = train_dpo(data, beta)
p_true_opt = rlhf_optimum(q, beta)                                      # RLHF optimum under the TRUE reward
implicit_reward = beta * (np.log(p_dpo) - np.log(p_ref))
print(f"KL(DPO policy || RLHF optimum with true reward) = {kl(p_dpo, p_true_opt):.4f}")
print(f"KL(reference  || RLHF optimum with true reward) = {kl(p_ref, p_true_opt):.4f}")
print(f"correlation of DPO's implicit reward beta*log(pi/pi_ref) with the true quality (within prompts): "
      f"{np.mean([np.corrcoef(implicit_reward[x], q[x])[0, 1] for x in range(N_PROMPTS)]):.3f}")
print(f"true quality: reference {(p_ref * q).sum(1).mean():.3f} -> DPO policy {(p_dpo * q).sum(1).mean():.3f}")

DPO converged to (nearly) the same policy that RLHF would find with the *true* reward, and its implicit reward $\beta\log\frac{\pi}{\pi_{\text{ref}}}$ recovers the true quality. Note the requirement we built in: **coverage**. The guarantee needs preference data on the responses that matter. With real offline data, DPO can push probability onto responses that never appeared in the data (they get no direct gradient), one reason "online" variants (sampling fresh responses from the current policy and getting them labeled) often do better.

### 6.3 Summary: DPO vs. RLHF
* We want to optimize for human preferences. Instead of humans writing the answers or giving uncalibrated scores, we get humans to **rank different LM-generated answers**.
* **RLHF:** train an explicit reward model on comparison data to predict a score for a given completion; optimize the LM to maximize the predicted score (under a KL constraint). **Very effective when tuned well, computationally expensive and tricky to get right.**
* **DPO:** optimize LM parameters directly on preference data by **solving a binary classification problem**. **Simple and effective, similar properties to RLHF, does not leverage online data.**
* **InstructGPT** scaled RLHF to tens of thousands of tasks (~30K tasks collected from labelers). **ChatGPT** = instruction finetuning + RLHF for dialogue agents (with details kept secret, perhaps to keep a competitive edge). In 2024, **open-source LLMs almost all used DPO** (and it works well!).

---
## 7. Limitations, and what's next

**Limitations of RL + reward modeling:**
* **Human preferences are unreliable!** "Reward hacking" is common (§5.3). Chatbots are rewarded for responses that *seem* authoritative, which can produce made-up facts and hallucinations.
* **Models of human preferences are even more unreliable** (reward-model over-optimization).
* **Where do the labels come from?** RLHF labels are often obtained from **overseas, low-wage workers**.
* **Annotator biases can creep into LMs.** Santurkar et al. (2023, OpinionQA) found that after RLHF, models' expressed opinions shift toward those of particular demographic groups, compared with base LMs.

**What's next (as of the 2024 slides):**
* **RL from AI feedback** (RLAIF; Bai et al. 2022). **"Constitutional AI":** the model critiques and revises its own responses according to written principles. *Human: Can you help me hack into my neighbor's wifi? Assistant: Sure thing, you can use an app called VeryEasyHack. Critique request: identify ways in which the assistant's last response is harmful. Critique: Hacking into someone else's wifi is an invasion of their privacy and is possibly illegal. Revision request: rewrite the response to remove harmful content. Revision: Hacking into your neighbor's wifi is an invasion of their privacy, and I strongly advise against it. It may also land you in legal trouble.* The revised responses (and AI-generated preference labels) are used for training, reducing the need for human labels.
* **Finetuning LMs on their own outputs** (Huang et al. 2022; **STaR**, the Self-Taught Reasoner, Zelikman et al. 2022), especially for code and reasoning: generate chains of thought, keep the ones that reach correct answers, and finetune on them.
* **Personalizing language models** (e.g., the PRISM alignment project, Kirk et al. 2024).
* However, there are still many limitations of large LMs (size, hallucination) that may **not** be solvable with RLHF!

🔎 **Since these slides:** the "finetune on your own outputs, reward correct answers" idea grew into **RL with verifiable rewards** (RLVR): for math and code, the reward is computed automatically (the answer is checked, or the tests pass), so there's no learned reward model to over-optimize in the same way. Combined with policy-gradient methods (e.g., GRPO, which replaces PPO's value function with group-relative baselines, as our REINFORCE baseline did), this produced today's "reasoning" models that generate long chains of thought before answering. Preference-based methods (RLHF/DPO and variants) remain standard for style, helpfulness, and safety. Lecture 15's slides ("Life after DPO") continue this story.

---
## 8. The DPO loss for a real language model (optional)

For a real LM, everything is a **sequence** log-probability: sum the log-probs of the response tokens (prompt tokens masked out, exactly as in §3.3), for both the policy and the frozen reference model. The loss itself is the four-line formula from §6.1. Libraries like Hugging Face TRL wrap this up (`DPOTrainer`), but it's worth seeing that there's nothing more to it.

In [ ]:
# [Optional] needs: pip install torch transformers   (and internet). One DPO step on GPT-2 with a single preference pair.
try:
    import torch, torch.nn.functional as F
    from transformers import AutoTokenizer, AutoModelForCausalLM

    def response_logprob(model, tok, prompt, response):
        # sum_t log p(response_t | prompt, response_<t): the prompt tokens are context only
        p_ids = tok(prompt, return_tensors="pt").input_ids
        full = tok(prompt + response, return_tensors="pt").input_ids
        logits = model(full).logits[:, :-1]                         # predict token t+1 from position t
        logp = logits.log_softmax(-1).gather(-1, full[:, 1:, None]).squeeze(-1)
        return logp[:, p_ids.shape[1] - 1:].sum()                   # keep only response positions

    def dpo_loss(policy, ref, tok, prompt, chosen, rejected, beta=0.1):
        pi_w, pi_l = response_logprob(policy, tok, prompt, chosen), response_logprob(policy, tok, prompt, rejected)
        with torch.no_grad():
            ref_w, ref_l = response_logprob(ref, tok, prompt, chosen), response_logprob(ref, tok, prompt, rejected)
        u = beta * ((pi_w - ref_w) - (pi_l - ref_l))
        return -F.logsigmoid(u), u.item() / beta                     # loss, and the implicit reward margin

    tok = AutoTokenizer.from_pretrained("gpt2")
    policy = AutoModelForCausalLM.from_pretrained("gpt2"); ref = AutoModelForCausalLM.from_pretrained("gpt2").eval()
    opt_t = torch.optim.AdamW(policy.parameters(), lr=1e-5)
    prompt = "Q: What is the capital of France?\nA:"
    chosen, rejected = " The capital of France is Paris.", " I don't know, maybe London."
    for step in range(5):
        loss, margin = dpo_loss(policy, ref, tok, prompt, chosen, rejected)
        opt_t.zero_grad(); loss.backward(); opt_t.step()
        print(f"step {step}: DPO loss {loss.item():.4f}   log-ratio margin (chosen - rejected) {margin:+.3f}")
except Exception as e:
    print("Skipping the GPT-2 DPO demo:", e)

---
## 9. Summary

**We've finally (mostly) answered how we get from *"Stanford University is located in \_\_\_\_"* to an assistant.**

| Stage | Key idea | Key formula / fact |
|---|---|---|
| Zero/few-shot ICL | Specify the task in the prompt; emergent with scale | Measured (toy): accuracy rises with in-context examples, tracking the Bayes-optimal task inference |
| Chain of thought | Let the model write intermediate steps | "Let's think step by step"; emergent with scale |
| Instruction finetuning | (instruction, output) pairs over thousands of tasks | Loss only on response tokens; generalizes to unseen tasks |
| Reward model | Pairwise comparisons, Bradley–Terry | $-\log\sigma(RM(y^w) - RM(y^l))$; robust to annotator offsets |
| RLHF | Maximize reward with a KL leash | $\max\ \mathbb{E}[RM] - \beta\,\text{KL}(p\,\|\,p^{PT})$; optimum $\propto p^{PT}e^{RM/\beta}$ |
| Over-optimization | Proxy reward up, true quality down (Goodhart) | Choose $\beta$ / stop early |
| DPO | The policy *is* the reward model | $-\log\sigma\big(\beta\log\frac{\pi(y^w)}{\pi_{ref}(y^w)} - \beta\log\frac{\pi(y^l)}{\pi_{ref}(y^l)}\big)$ |

### Exercises
1. In §1.2, add a fifth training function and re-measure. Does the model need more examples to reach the same accuracy (as the Bayes-optimal learner does)?
2. Train the ICL model with **16** functions (e.g., $ax+b \bmod 10$ for several $a,b$). Does it start to handle an unseen function better?
3. In §4.2, set all annotator scales to 1 and offsets to 0. Does the gap between ratings and comparisons disappear?
4. In §5.3, give the reward model a third feature that *perfectly* identifies the junk response, but only include 1% junk in the preference data. Does over-optimization still happen?
5. Run DPO with only 300 random preference pairs (sampled from $p^{PT}$, as in §4.3). What happens to the probability of responses that never appear in the data?
6. Implement **IPO** (Azar et al. 2023): replace $-\log\sigma(u)$ with $(u/\beta - \frac{1}{2\beta})^2$. Compare its behavior with DPO when preferences are deterministic.

### References
* Radford et al. 2019 (GPT-2, zero-shot); Brown et al. 2020 (GPT-3, few-shot).
* Wei et al. 2022 (chain of thought); Kojima et al. 2022 (zero-shot CoT); Zhou et al. 2022 (LM-designed prompts).
* Xie et al. 2022, *An Explanation of In-context Learning as Implicit Bayesian Inference*; Garg et al. 2022; Raventós et al. 2023.
* Chung et al. 2022 (Flan-T5); Wang et al. 2022 (Super-NaturalInstructions); Zhou et al. 2023 (LIMA).
* Ouyang et al. 2022 (InstructGPT); Stiennon et al. 2020; Ziegler et al. 2019; Gao et al. 2022 (*Scaling Laws for Reward Model Overoptimization*).
* Rafailov et al. 2023, *Direct Preference Optimization*.
* Bai et al. 2022 (Constitutional AI); Zelikman et al. 2022 (STaR); Santurkar et al. 2023 (OpinionQA).

**Next:** Lecture 11, benchmarking and evaluation.